In [7]:


import numpy as np
import xarray as xr
import metpy.calc as mpcalc
import time
import os
import glob
from metpy.units import units
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


FIG_SAVE_DIR = "./figures/divergence/"

# 创建必要的目录
for d in [ FIG_SAVE_DIR]:
    os.makedirs(d, exist_ok=True)


EXPERIMENTS = ['CNTL', 'P4K', '4CO2']

In [8]:


# ============ 数据处理函数 ============
def load_data(path: str, var: str, lat_range: tuple = (-15, 15)) -> xr.DataArray:
    """加载并预处理数据"""
    ds = xr.open_dataset(path).sortby('lat').sel(lat=slice(*lat_range))
    return ds[var]

def interpolate_nan(da: xr.DataArray) -> xr.DataArray:
    """填充 NaN 值"""
    return (da.interpolate_na(dim='time', method='nearest')
              .interpolate_na(dim='lat', method='nearest')
              .interpolate_na(dim='lon', method='nearest'))

def compute_grid_spacing(lat: np.ndarray, lon: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """计算网格间距 (m)"""
    R = 6.371e6  # 地球半径
    xlon, ylat = np.meshgrid(lon, lat)
    dlonx = np.gradient(xlon, axis=1)
    dlaty = np.gradient(ylat, axis=0)
    dx = R * np.cos(ylat * np.pi / 180) * dlonx * np.pi / 180
    dy = R * dlaty * np.pi / 180
    return dx, dy

def numpy_divergence(u: xr.DataArray, v: xr.DataArray, dx: np.ndarray, dy: np.ndarray) -> np.ndarray:
    """使用 NumPy 计算散度"""
    du_dx = np.gradient(u, axis=-1) * units('m/s') / dx[np.newaxis, :, :]
    dv_dy = np.gradient(v, axis=-2) * units('m/s') / dy[np.newaxis, :, :]
    return (du_dx + dv_dy).magnitude

def find_matching_pressure_level(available_levels: xr.DataArray, target_level: float) -> float:
    """找到匹配的压力层"""
    matched = [p.item() for p in available_levels if np.isclose(p, target_level, atol=1)]
    if not matched:
        raise ValueError(f"未找到匹配的压力层：{target_level}")
    return matched[0]

# ============ 主要处理函数 ============
def calculate_divergence(ua_path: str, va_path: str, plev: float) -> Dict[str, xr.DataArray]:
    """计算 MetPy 和 NumPy 散度"""
    print(f"  📌 计算散度 (plev={plev/100:.0f}hPa)")
    
    # 加载数据
    ds = xr.open_dataset(ua_path)
    matched_plev = find_matching_pressure_level(ds.plev, plev)
    
    ua = load_data(ua_path, 'ua', CONFIG['lat_range']).sel(plev=matched_plev)
    va = load_data(va_path, 'va', CONFIG['lat_range']).sel(plev=matched_plev)
    
    # 预处理
    ua_clean = interpolate_nan(ua) * units.meter / units.second
    va_clean = interpolate_nan(va) * units.meter / units.second
    
    # 计算网格间距
    dx, dy = compute_grid_spacing(ua.lat.values, ua.lon.values)
    
    # 计算散度
    div_metpy = mpcalc.divergence(ua_clean, va_clean)
    div_numpy = numpy_divergence(ua_clean, va_clean, dx * units.meter, dy * units.meter)
    
    # 创建 DataArray，去掉 plev 坐标避免冲突
    coords_no_plev = {k: v for k, v in ua_clean.coords.items() if k != 'plev'}
    return {
        'metpy': xr.DataArray(div_metpy, dims=('time', 'lat', 'lon'), coords=coords_no_plev),
        'numpy': xr.DataArray(div_numpy, dims=('time', 'lat', 'lon'), coords=coords_no_plev)
    }

def get_file_path(base_dir: str, var: str, model: str) -> str:
    """获取文件路径"""
    pattern = os.path.join(base_dir, var, "sametime", f"{var}_day_{model}_*.nc")
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"未找到文件：{pattern}")
    if len(files) > 1:
        print(f"  ⚠️ 找到多个文件，使用第一个：{files[0]}")
    return files[0]

def save_divergence_data(divergence_results: Dict[str, Dict[str, xr.DataArray]], 
                        output_path: str) -> None:
    """保存散度数据"""
    datasets = {}
    for plev, div_data in divergence_results.items():
        # 为每个变量添加压力层属性
        metpy_da = div_data['metpy'].copy()
        numpy_da = div_data['numpy'].copy()
        
        # 添加压力层作为属性而不是坐标
        metpy_da.attrs['pressure_level'] = f'{plev}Pa'
        numpy_da.attrs['pressure_level'] = f'{plev}Pa'
        
        datasets[f'divergence_metpy_{plev}'] = metpy_da
        datasets[f'divergence_numpy_{plev}'] = numpy_da
    
    ds_out = xr.Dataset(datasets)
    
    # 添加全局属性
    ds_out.attrs['description'] = 'Divergence calculated using MetPy and NumPy methods'
    ds_out.attrs['pressure_levels'] = f'{list(divergence_results.keys())}Pa'
    ds_out.attrs['created_by'] = 'Improved divergence calculation script'
    
    ds_out.to_netcdf(output_path)
    print(f"  ✅ 数据已保存：{output_path}")

def plot_divergence_comparison(nc_path: str, plev: float, save_plot: bool = True) -> None:
    """绘制散度对比图"""
    ds = xr.open_dataset(nc_path)
    model = os.path.basename(nc_path).split("_")[0]
    
    div_metpy = ds[f'divergence_metpy_{plev}'].mean(dim='time')
    div_numpy = ds[f'divergence_numpy_{plev}'].mean(dim='time')
    diff = div_metpy - div_numpy
    
    # 创建图形
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), 
                            subplot_kw={'projection': ccrs.PlateCarree()})
    
    vmax = max(np.abs(div_metpy.max()), np.abs(div_metpy.min()))
    titles = ["MetPy Divergence", "NumPy Divergence", "Difference (MetPy - NumPy)"]
    data_list = [div_metpy, div_numpy, diff]
    vmaxs = [vmax, vmax, vmax/10]
    
    for i, (ax, title, data, vm) in enumerate(zip(axes, titles, data_list, vmaxs)):
        cmap = 'RdBu_r' if i < 2 else 'RdBu'
        im = ax.pcolormesh(data.lon, data.lat, data, cmap=cmap, vmin=-vm, vmax=vm)
        ax.set_title(f"{title} ({model}, {plev/100:.0f}hPa)")
        ax.coastlines()
        ax.set_xticks(np.arange(0, 361, 60), crs=ccrs.PlateCarree())
        ax.set_yticks(np.arange(-15, 16, 5), crs=ccrs.PlateCarree())
        fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.05)
    
    plt.tight_layout()
    
    if save_plot:
        plot_path = nc_path.replace('.nc', f'_{plev//100}hPa_comparison.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        print(f"  📊 图像已保存：{plot_path}")
    
    plt.show()
    
    # 统计分析
    print_statistics(ds[f'divergence_metpy_{plev}'], ds[f'divergence_numpy_{plev}'], plev)

def print_statistics(metpy_data: xr.DataArray, numpy_data: xr.DataArray, plev: float) -> None:
    """打印统计信息"""
    m, n = metpy_data.values.flatten(), numpy_data.values.flatten()
    valid = np.isfinite(m) & np.isfinite(n)
    
    if valid.sum() == 0:
        print(f"  ❌ {plev/100:.0f}hPa: 无有效数据进行比较")
        return
    
    corr, _ = pearsonr(m[valid], n[valid])
    rmse = np.sqrt(mean_squared_error(m[valid], n[valid]))
    bias = np.mean(m[valid] - n[valid])
    
    print(f"  📈 {plev/100:.0f}hPa 统计:")
    print(f"     🔹 相关系数: {corr:.4f}")
    print(f"     🔹 RMSE: {rmse:.4e}")
    print(f"     🔹 Bias: {bias:.4e}")

def extract_model_name(file_path: str) -> str:
    """从文件路径提取模型名称"""
    return os.path.basename(file_path).split("_")[2]

        

In [9]:
# 1. 加载 zg 数据（位势高度，单位：m）
print("读取 zg 数据...")
zg_cntl = xr.open_dataset('processed_data/zg_2deg_interp_cntl.nc')['zg']
zg_p4k = xr.open_dataset('processed_data/zg_2deg_interp_p4k.nc')['zg']
zg_4co2 = xr.open_dataset('processed_data/zg_2deg_interp_4co2.nc')['zg']

# 2. 检查维度名称并标准化
print(f"  zg_cntl dimensions: {zg_cntl.dims}")
if 'level_full' in zg_cntl.dims:
    zg_cntl = zg_cntl.rename({'level_full': 'lev'})
    zg_p4k = zg_p4k.rename({'level_full': 'lev'})
    zg_4co2 = zg_4co2.rename({'level_full': 'lev'})
    print("  ✓ 维度 'level_full' 已重命名为 'lev'")

# 3. 创建海洋 mask（如果还没有的话）
try:
    # 尝试使用已有的 ocean_mask
    test_mask = ocean_mask
    print("✓ 使用已有的 ocean_mask")
except NameError:
    # 如果不存在，创建一个简单的海洋 mask（全部为True，即所有点都计算）
    print("创建海洋 mask...")
    ocean_mask = xr.ones_like(zg_cntl.isel(lev=0, drop=True))
    print("✓ 海洋 mask 创建完成")

# 4. 应用海洋 mask
zg_cntl_ocean = zg_cntl.where(ocean_mask)
zg_p4k_ocean = zg_p4k.where(ocean_mask)
zg_4co2_ocean = zg_4co2.where(ocean_mask)

# 5. 计算海洋平均高度（对每一层）
# zg 不随时间变化，所以只对空间平均
zg_cntl_mean = zg_cntl_ocean.mean(dim=['lat', 'lon'], skipna=True)
zg_p4k_mean = zg_p4k_ocean.mean(dim=['lat', 'lon'], skipna=True)
zg_4co2_mean = zg_4co2_ocean.mean(dim=['lat', 'lon'], skipna=True)

# 6. 转换为 km 单位
zg_cntl_km = zg_cntl_mean / 1000.0
zg_p4k_km = zg_p4k_mean / 1000.0
zg_4co2_km = zg_4co2_mean / 1000.0

# 7. 创建高度字典
zg_dict = {
    'CNTL': zg_cntl_km,
    'P4K': zg_p4k_km,
    '4CO2': zg_4co2_km
}

print(f"\n✓ zg 数据加载完成")
print(f"  高度范围: {zg_cntl_km.min().values:.2f} - {zg_cntl_km.max().values:.2f} km")
print(f"  层数: {len(zg_cntl_km)}")
print(f"  维度: {zg_cntl_km.dims}")

读取 zg 数据...
  zg_cntl dimensions: ('level_full', 'lat', 'lon')
  ✓ 维度 'level_full' 已重命名为 'lev'
✓ 使用已有的 ocean_mask

✓ zg 数据加载完成
  高度范围: 0.13 - 22.86 km
  层数: 22
  维度: ('lev',)


In [10]:
zg_cntl_km

<xarray.DataArray 'zg' (lev: 22)> Size: 176B
array([22.86439757, 19.53890208, 17.52363904, 15.8434868 , 13.61785013,
       11.61962614, 10.02229384,  8.82550181,  6.83467525,  5.25165022,
        3.81291878,  2.88034656,  2.32814921,  1.83180114,  1.39191719,
        1.19348922,  0.84077287,  0.68714422,  0.54928321,  0.32372966,
        0.1736692 ,  0.13446286])
Coordinates:
  * lev      (lev) float64 176B 31.0 35.0 38.0 41.0 46.0 ... 85.0 87.0 89.0 90.0

In [11]:
zg_p4k_km

<xarray.DataArray 'zg' (lev: 22)> Size: 176B
array([22.86439757, 19.53890208, 17.52363904, 15.8434868 , 13.61785013,
       11.61962614, 10.02229384,  8.82550181,  6.83467525,  5.25165022,
        3.81291878,  2.88034656,  2.32814921,  1.83180114,  1.39191719,
        1.19348922,  0.84077287,  0.68714422,  0.54928321,  0.32372966,
        0.1736692 ,  0.13446286])
Coordinates:
  * lev      (lev) float64 176B 31.0 35.0 38.0 41.0 46.0 ... 85.0 87.0 89.0 90.0

In [12]:
# ============ 多层散度计算（优化版：矩阵计算 + 仅海洋） ============

def calculate_low_level_divergence_multi_levels(target_levels=[80, 81, 55, 50]):
    """
    计算多个层次的散度
    
    Parameters:
    -----------
    target_levels : list
        要计算的层次列表，例如 [80, 81, 55, 50]
        Level 80-81: ~850hPa (低层)
        Level 55: ~500hPa (中层)
        Level 50: ~400hPa (中高层)
    
    优化策略：
    1. 使用已有的 numpy_divergence 函数进行矩阵计算，避免循环
    2. 只计算海洋区域的数据，节省计算资源
    3. 支持多个层次的批量计算
    """
    print("=" * 60)
    print(f"开始计算多层散度（优化版）")
    print(f"目标层次: {target_levels}")
    print("=" * 60)
    
    # 1. 加载海洋mask
    print(f"\n🌊 加载海洋mask...")
    try:
        ocean_mask_file = 'processed_data/ocean_mask_2deg.nc'
        ocean_mask_ds = xr.open_dataset(ocean_mask_file)
        ocean_mask = ocean_mask_ds['ocean_mask']
        print(f"   ✅ 从文件加载海洋mask: {ocean_mask_file}")
    except:
        print(f"   ⚠️ 未找到海洋mask文件，使用全球数据")
        ocean_mask = None
    
    # 实验配置
    experiments = {
        'cntl': {'ua_dir': 'ua_cntl_layers', 'va_dir': 'va_cntl_layers'},
        'p4k': {'ua_dir': 'ua_p4k_layers', 'va_dir': 'va_p4k_layers'},
        '4co2': {'ua_dir': 'ua_4co2_layers', 'va_dir': 'va_4co2_layers'}
    }
    
    # 存储结果：嵌套字典 {exp_name: {level: divergence_data}}
    divergence_results = {exp: {} for exp in experiments.keys()}
    
    # 循环处理每个层次
    for target_level in target_levels:
        print(f"\n{'='*60}")
        print(f"📍 处理层次: Level {target_level}")
        print(f"{'='*60}")
        
        for exp_name, exp_info in experiments.items():
  
            print(f"\n{'─'*50}")
            print(f"处理实验: {exp_name.upper()} - Level {target_level}")
            print(f"{'─'*50}")
            
            # 构建文件路径
            ua_file = f"{exp_info['ua_dir']}/ua_lev_{target_level:03d}.nc"
            va_file = f"{exp_info['va_dir']}/va_lev_{target_level:03d}.nc"
            
            print(f"📂 ua文件: {ua_file}")
            print(f"📂 va文件: {va_file}")
            
            # 检查文件是否存在
            if not os.path.exists(ua_file) or not os.path.exists(va_file):
                print(f"❌ 文件不存在，跳过 {exp_name} - Level {target_level}")
                continue
            
            # 1. 加载数据
            print(f"\n1️⃣ 加载风场数据...")
            ua_ds = xr.open_dataset(ua_file)
            va_ds = xr.open_dataset(va_file)
            
            ua = ua_ds['ua']
            va = va_ds['va']
            
            print(f"   ua shape: {ua.shape}, dims: {ua.dims}")
            print(f"   va shape: {va.shape}, dims: {va.dims}")
            
            # 2. 去掉level维度（只有一层）并选择热带区域 (-15°N to 15°N)
            print(f"\n2️⃣ 去掉level维度并选择热带区域 (-15°N to 15°N)...")
            # 去掉level维度
            ua = ua.squeeze('level', drop=True) if 'level' in ua.dims else ua
            va = va.squeeze('level', drop=True) if 'level' in va.dims else va
            
            ua_tropical = ua.sel(lat=slice(-15, 15))
            va_tropical = va.sel(lat=slice(-15, 15))
            
            print(f"   热带区域 ua shape: {ua_tropical.shape}")
            print(f"   热带区域 ua dims: {ua_tropical.dims}")
            
            # 3. 应用海洋mask（只计算海洋区域）
            if ocean_mask is not None:
                print(f"\n3️⃣ 应用海洋mask...")
                # 确保mask与数据维度匹配
                ocean_mask_tropical = ocean_mask.sel(lat=slice(-15, 15))
                
                # 广播mask到时间维度
                ocean_mask_broadcast = ocean_mask_tropical.broadcast_like(ua_tropical)
                
                # 只保留海洋区域的数据
                ua_ocean = ua_tropical.where(ocean_mask_broadcast)
                va_ocean = va_tropical.where(ocean_mask_broadcast)
                
                ocean_points = ocean_mask_tropical.sum().values
                total_points = ocean_mask_tropical.size
                print(f"   海洋点数: {ocean_points} / {total_points} ({100*ocean_points/total_points:.1f}%)")
            else:
                ua_ocean = ua_tropical
                va_ocean = va_tropical
            
            # 4. 数据预处理：填充NaN值
            print(f"\n4️⃣ 填充NaN值...")
            ua_clean = interpolate_nan(ua_ocean)
            va_clean = interpolate_nan(va_ocean)
            
            print(f"   填充后 ua shape: {ua_clean.shape}")
            
            # 5. 计算网格间距
            print(f"\n5️⃣ 计算网格间距...")
            dx, dy = compute_grid_spacing(ua_clean.lat.values, ua_clean.lon.values)
            print(f"   dx range: {dx.min():.1f} - {dx.max():.1f} m")
            print(f"   dy range: {dy.min():.1f} - {dy.max():.1f} m")
            
            # 6. 使用矩阵计算散度（调用已有函数，无需循环）
            print(f"\n6️⃣ 计算散度（矩阵计算，无循环）...")
            print(f"   使用 numpy_divergence 函数进行高效计算...")
            
            # 直接使用已有的 numpy_divergence 函数进行矩阵计算
            divergence = numpy_divergence(ua_clean, va_clean, dx, dy)
            
            print(f"   ✅ 散度计算完成")
            print(f"   散度 shape: {divergence.shape}")
            print(f"   散度范围: {np.nanmin(divergence):.2e} - {np.nanmax(divergence):.2e} s^-1")
            
            # 7. 创建DataArray
            print(f"\n7️⃣ 创建DataArray...")
            div_da = xr.DataArray(
                divergence,
                dims=('time', 'lat', 'lon'),
                coords={
                    'time': ua_clean.time,
                    'lat': ua_clean.lat,
                    'lon': ua_clean.lon
                },
                attrs={
                    'long_name': f'Horizontal divergence at Level {target_level} - Ocean only',
                    'units': 's^-1',
                    'level': target_level,
                    'method': 'NumPy gradient method (vectorized)',
                    'domain': 'Ocean only' if ocean_mask is not None else 'Global',
                    'experiment': exp_name.upper()
                }
            )
            
            # 如果有ocean mask，将陆地区域设为NaN
            if ocean_mask is not None:
                ocean_mask_broadcast = ocean_mask_tropical.broadcast_like(div_da)
                div_da = div_da.where(ocean_mask_broadcast)
            
            divergence_results[exp_name][target_level] = div_da
            
            print(f"✅ {exp_name.upper()} - Level {target_level} 散度计算完成")
    
    return divergence_results, target_levels


# 执行计算
print("\n开始执行多层散度计算（优化版）...")
print("优化特点：")
print("  ✓ 使用矩阵运算，避免时间循环")
print("  ✓ 仅计算海洋区域数据")
print("  ✓ 调用已有的 numpy_divergence 函数")
print("  ✓ 支持多个层次批量计算")

# 定义要计算的层次
TARGET_LEVELS = [80, 81, 55, 51]

start_time = time.time()

divergence_multi_levels, levels_used = calculate_low_level_divergence_multi_levels(TARGET_LEVELS)

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"✅ 计算完成！总耗时: {elapsed:.2f} 秒")
print(f"✅ 计算的层次: {levels_used}")
print(f"{'='*60}")


开始执行多层散度计算（优化版）...
优化特点：
  ✓ 使用矩阵运算，避免时间循环
  ✓ 仅计算海洋区域数据
  ✓ 调用已有的 numpy_divergence 函数
  ✓ 支持多个层次批量计算
开始计算多层散度（优化版）
目标层次: [80, 81, 55, 51]

🌊 加载海洋mask...
   ⚠️ 未找到海洋mask文件，使用全球数据

📍 处理层次: Level 80

──────────────────────────────────────────────────
处理实验: CNTL - Level 80
──────────────────────────────────────────────────
📂 ua文件: ua_cntl_layers/ua_lev_080.nc
📂 va文件: va_cntl_layers/va_lev_080.nc

1️⃣ 加载风场数据...
   ua shape: (1, 5114, 15, 180), dims: ('level', 'time', 'lat', 'lon')
   va shape: (1, 5114, 15, 180), dims: ('level', 'time', 'lat', 'lon')

2️⃣ 去掉level维度并选择热带区域 (-15°N to 15°N)...
   热带区域 ua shape: (5114, 15, 180)
   热带区域 ua dims: ('time', 'lat', 'lon')

4️⃣ 填充NaN值...
   填充后 ua shape: (5114, 15, 180)

5️⃣ 计算网格间距...
   dx range: 215783.9 - 222389.9 m
   dy range: 222389.9 - 222389.9 m

6️⃣ 计算散度（矩阵计算，无循环）...
   使用 numpy_divergence 函数进行高效计算...
   ✅ 散度计算完成
   散度 shape: (5114, 15, 180)
   散度范围: -1.78e-04 - 1.22e-04 s^-1

7️⃣ 创建DataArray...
✅ CNTL - Level 80 散度计算完成

──────────────────

In [13]:
# ============ 保存多层散度数据 ============

def save_multi_level_divergence(divergence_results, levels_used, output_dir='processed_data'):
    """
    保存多层散度数据到NetCDF文件
    
    Parameters:
    -----------
    divergence_results : dict
        嵌套字典 {exp_name: {level: divergence_data}}
    levels_used : list
        使用的层次列表
    output_dir : str
        输出目录
    """
    print("\n" + "="*60)
    print("保存多层散度数据")
    print(f"层次: {levels_used}")
    print("="*60)
    
    os.makedirs(output_dir, exist_ok=True)
    
    saved_files = []
    
    for exp_name, level_dict in divergence_results.items():
        for level, div_data in level_dict.items():
            # 构建输出文件名
            output_file = os.path.join(
                output_dir, 
                f'divergence_lev{level}_{exp_name}.nc'
            )
            
            print(f"\n📝 保存 {exp_name.upper()} Level {level} 数据...")
            print(f"   文件: {output_file}")
            
            # 创建数据集
            ds_out = xr.Dataset({
                'divergence': div_data
            })
            
            # 添加全局属性
            ds_out.attrs.update({
                'title': f'Divergence at Level {level} for {exp_name.upper()} experiment',
                'level': level,
                'experiment': exp_name.upper(),
                'method': 'NumPy gradient method',
                'spatial_domain': 'Tropical (15°S-15°N)',
                'created_date': time.strftime('%Y-%m-%d %H:%M:%S'),
                'units': 's^-1',
                'description': 'Horizontal wind divergence calculated from ua and va wind components'
            })
            
            # 保存到文件
            ds_out.to_netcdf(output_file)
            saved_files.append(output_file)
            
            # 打印数据摘要
            print(f"   ✅ 保存成功")
            print(f"   数据形状: {div_data.shape}")
            print(f"   时间范围: {div_data.time.values[0]} to {div_data.time.values[-1]}")
            print(f"   空间范围: lat [{div_data.lat.min().values:.1f}, {div_data.lat.max().values:.1f}], "
                  f"lon [{div_data.lon.min().values:.1f}, {div_data.lon.max().values:.1f}]")
            print(f"   散度范围: [{np.nanmin(div_data.values):.2e}, {np.nanmax(div_data.values):.2e}] s^-1")
            
            # 关闭数据集
            ds_out.close()
    
    print(f"\n{'='*60}")
    print(f"✅ 所有数据已保存！共 {len(saved_files)} 个文件")
    print(f"{'='*60}")
    
    return saved_files


# 保存数据
print("\n保存计算结果...")
saved_files = save_multi_level_divergence(divergence_multi_levels, levels_used, output_dir='processed_data')

print("\n📁 已保存的文件:")
for f in saved_files:
    print(f"   ✓ {f}")


保存计算结果...

保存多层散度数据
层次: [80, 81, 55, 51]

📝 保存 CNTL Level 80 数据...
   文件: processed_data/divergence_lev80_cntl.nc


PermissionError: [Errno 13] Permission denied: '/work/mh1498/m301257/processed_data/divergence_lev80_cntl.nc'

## 验证和可视化多层散度

快速验证保存的数据并绘制平均散度场（支持多个层次）

In [ ]:
# ============ 验证和可视化多层散度 ============

def verify_and_plot_multi_level_divergence(saved_files, levels_to_plot=None):
    """
    验证保存的数据并绘制平均散度场
    
    Parameters:
    -----------
    saved_files : list
        保存的文件列表
    levels_to_plot : list, optional
        要绘制的层次列表，如果为None则绘制所有层次
    """
    print("\n" + "="*60)
    print("验证和可视化多层散度数据")
    print("="*60)
    
    # 从文件名提取层次信息
    available_levels = set()
    for f in saved_files:
        # 文件名格式: divergence_lev{level}_{exp}.nc
        import re
        match = re.search(r'lev(\d+)', f)
        if match:
            available_levels.add(int(match.group(1)))
    
    available_levels = sorted(list(available_levels))
    print(f"\n可用的层次: {available_levels}")
    
    if levels_to_plot is None:
        levels_to_plot = available_levels
    else:
        levels_to_plot = [lev for lev in levels_to_plot if lev in available_levels]
    
    print(f"绘制的层次: {levels_to_plot}")
    
    experiments_order = ['cntl', 'p4k', '4co2']
    titles = ['CNTL', 'P4K', '4CO2']
    
    # 为每个层次创建一个图
    for level in levels_to_plot:
        print(f"\n{'='*60}")
        print(f"绘制 Level {level}")
        print(f"{'='*60}")
        
        # 创建图形
        fig, axes = plt.subplots(3, 1, figsize=(10, 15),
                                subplot_kw={'projection': ccrs.PlateCarree(180)})
        
        for idx, (exp_name, ax, title) in enumerate(zip(experiments_order, axes, titles)):
            # 查找对应的文件
            exp_file = [f for f in saved_files 
                       if exp_name in f.lower() and f'lev{level}' in f]
            
            if not exp_file:
                print(f"⚠️ 未找到 {exp_name} Level {level} 的数据文件")
                continue
            
            exp_file = exp_file[0]
            print(f"\n📊 绘制 {exp_name.upper()} Level {level}...")
            print(f"   文件: {exp_file}")
            
            # 读取数据
            ds = xr.open_dataset(exp_file)
            div = ds['divergence']
            
            # 计算时间平均
            div_mean = div.mean(dim='time')
            
            # 打印统计信息
            print(f"   时间平均散度统计:")
            print(f"     最小值: {div_mean.min().values:.2e} s^-1")
            print(f"     最大值: {div_mean.max().values:.2e} s^-1")
            print(f"     平均值: {div_mean.mean().values:.2e} s^-1")
            print(f"     标准差: {div_mean.std().values:.2e} s^-1")
            
            # 绘图
            vmax = max(abs(div_mean.min().values), abs(div_mean.max().values))
            vmax = min(vmax, 5e-6)  # 限制色标范围以便更好地显示
            
            im = ax.pcolormesh(
                div_mean.lon, 
                div_mean.lat, 
                div_mean,
                cmap='RdBu_r',
                vmin=-vmax,
                vmax=vmax,
                transform=ccrs.PlateCarree()
            )
            
            ax.coastlines()
            ax.set_title(f'{title} - Level {level} Divergence (Time Mean)', 
                        fontsize=12, fontweight='bold')
            ax.set_xlabel('Longitude')
            ax.set_ylabel('Latitude')
            ax.set_aspect('auto')
            # 添加网格
            ax.gridlines(draw_labels=False, linestyle='--', alpha=0.5)
            
            # 设置经纬度刻度
            ax.set_xticks(np.arange(0, 361, 60), crs=ccrs.PlateCarree())
            ax.set_yticks(np.arange(-15, 16, 5), crs=ccrs.PlateCarree())
            
            # 添加色标
            cbar = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.8)
            cbar.set_label('Divergence (s$^{-1}$)', fontsize=10)
            cbar.formatter.set_powerlimits((-2, 2))
            cbar.update_ticks()
            
            ds.close()
        
        plt.tight_layout()
        
        # 保存图像
        plot_file = os.path.join(FIG_SAVE_DIR, f'divergence_lev{level}_comparison.png')
        plt.savefig(plot_file, dpi=300, bbox_inches='tight')
        print(f"\n📈 图像已保存: {plot_file}")
        
        plt.show()
    
    print(f"\n{'='*60}")
    print("✅ 验证和可视化完成")
    print(f"{'='*60}")


# 执行验证和可视化
if saved_files:
    # 可以指定要绘制的层次，例如只绘制80和81层
    # verify_and_plot_multi_level_divergence(saved_files, levels_to_plot=[80, 81])
    
    # 或者绘制所有层次
    verify_and_plot_multi_level_divergence(saved_files)
else:
    print("⚠️ 没有可用的数据文件进行验证")

## 快速加载函数

提供一个便捷函数，方便后续直接调用850hPa散度数据

In [ ]:
# ============ 快速加载多层散度数据 ============

def load_divergence_by_level(experiment='cntl', level=80, data_dir='processed_data'):
    """
    快速加载指定层次的散度数据
    
    Parameters:
    -----------
    experiment : str
        实验名称，可选: 'cntl', 'p4k', '4co2'
    level : int
        层次编号，例如: 80, 81, 55, 50
    data_dir : str
        数据目录
    
    Returns:
    --------
    xr.DataArray
        散度数据
    
    Example:
    --------
    >>> div_cntl_lev80 = load_divergence_by_level('cntl', 80)
    >>> div_p4k_lev55 = load_divergence_by_level('p4k', 55)
    """
    # 构建文件路径
    file_pattern = os.path.join(data_dir, f'divergence_lev{level}_{experiment}.nc')
    files = glob.glob(file_pattern)
    
    if not files:
        raise FileNotFoundError(f"未找到实验 {experiment} Level {level} 的散度数据文件: {file_pattern}")
    
    if len(files) > 1:
        print(f"⚠️ 找到多个文件，使用第一个: {files[0]}")
    
    file_path = files[0]
    print(f"📂 加载数据: {file_path}")
    
    # 加载数据
    ds = xr.open_dataset(file_path)
    div = ds['divergence']
    
    print(f"✅ 数据加载成功")
    print(f"   形状: {div.shape}")
    print(f"   维度: {div.dims}")
    print(f"   时间范围: {div.time.values[0]} to {div.time.values[-1]}")
    print(f"   空间范围: lat [{div.lat.min().values:.1f}, {div.lat.max().values:.1f}], "
          f"lon [{div.lon.min().values:.1f}, {div.lon.max().values:.1f}]")
    
    return div


def load_all_divergence(experiments=['cntl', 'p4k', '4co2'], 
                       levels=[80, 81, 55, 51], 
                       data_dir='processed_data'):
    """
    加载所有实验和所有层次的散度数据
    
    Parameters:
    -----------
    experiments : list
        实验列表
    levels : list
        层次列表
    data_dir : str
        数据目录
    
    Returns:
    --------
    dict
        嵌套字典 {exp: {level: divergence_data}}
    """
    print("\n" + "="*60)
    print("加载所有散度数据")
    print(f"实验: {experiments}")
    print(f"层次: {levels}")
    print("="*60)
    
    divergence_data = {exp: {} for exp in experiments}
    
    for exp in experiments:
        for level in levels:
            try:
                print(f"\n加载 {exp.upper()} Level {level}...")
                div = load_divergence_by_level(exp, level, data_dir)
                divergence_data[exp][level] = div
            except FileNotFoundError as e:
                print(f"❌ {e}")
    
    print(f"\n{'='*60}")
    total_loaded = sum(len(level_dict) for level_dict in divergence_data.values())
    print(f"✅ 成功加载 {total_loaded} 个数据文件")
    print(f"{'='*60}")
    
    return divergence_data


# 使用示例
print("\n" + "="*60)
print("使用示例:")
print("="*60)
print("""
# 加载单个实验、单个层次的散度数据:
div_cntl_lev80 = load_divergence_by_level('cntl', 80)
div_p4k_lev55 = load_divergence_by_level('p4k', 55)

# 加载所有实验和所有层次的散度数据:
all_div = load_all_divergence()

# 访问特定实验和层次:
div_cntl_80 = all_div['cntl'][80]
div_p4k_81 = all_div['p4k'][81]
div_4co2_55 = all_div['4co2'][55]

# 计算时间平均:
div_mean = all_div['cntl'][80].mean(dim='time')

# 计算区域平均:
div_tropical_mean = all_div['cntl'][80].mean(dim=['lat', 'lon'])

# 比较不同层次:
for level in [80, 81, 55, 50]:
    if level in all_div['cntl']:
        div_mean = all_div['cntl'][level].mean()
        print(f"Level {level}: {div_mean.values:.2e} s^-1")
""")